In [ ]:
import os
os.chdir('/ictstr01/home/icb/fatemehs.hashemig/codes/interpretable-ssl')

In [ ]:
from interpretable_ssl.trainers.swav import *

# functions

In [ ]:
from scib_metrics.benchmark import Benchmarker
import os
def get_scib(adata, emb_keys, batch_key, label_key):
  bm = Benchmarker(
          adata=adata,
          batch_key=batch_key,
          label_key=label_key,
          embedding_obsm_keys=emb_keys,  # evaluate the PCA space
      )

  bm.benchmark()               # runs neighbors, clustering, and metrics
  results = bm.get_results(min_max_scale=False)   # returns a tidy DataFrame
  return results

def save_append(df, save_dir, name, append):
  os.makedirs(os.path.dirname(save_dir), exist_ok=True)
  if append:
    saved_res = pd.read_csv(f'{save_dir}/{name}', index_col = 0)
    df = pd.concat([df, saved_res])
  df.to_csv(f'{save_dir}/{name}')
  return df
    
def save_scib(results, dataset_name, append=True):
  save_path = f'~/models/{dataset_name}/'
  
  results = results.drop(index='Metric Type')
  return save_append(results, save_path, 'scib.csv', append)

In [ ]:
sys.path.append("/home/icb/fatemehs.hashemig/Islander/src")
from scGraph import *

def get_scgraph(adata, obsm_keys, dataset_name, batch_key='study', label_key='cell_type', append = True):
    adata.write('tmp.h5ad')
    scgraph = scGraph(
            adata_path='tmp.h5ad',
            batch_key=batch_key,
            label_key=label_key,
        )
    scgr_res = scgraph.main(_obsm_list=obsm_keys)
    save_path = f'~/models/{dataset_name}/'

    return save_append(scgr_res, save_path, 'scgraph.csv', append)
   

In [ ]:
def save_metrics(adata, emb_keys, dataset, bk, lk, append):
    scgraph = get_scgraph(adata, emb_keys, dataset, bk, lk, append)
    metrics = get_scib(adata, emb_keys, bk, lk)
    metrics = save_scib(metrics, dataset, append)
    return metrics, scgraph


# used to be save_metrics
def save_trainer_metrics(t, dataset, append = True):
    model = t.load_model()
    adata = t.dataset.adata
    adata.obsm[t.get_model_name()] = t.encode_adata(adata, model).detach().cpu().numpy()
    sc.tl.pca(adata)
    bk, lk = t.dataset.batch_key, t.dataset.cell_type_key
    return save_metrics(adata, [t.get_model_name()], dataset, bk, lk, append)


In [ ]:
import scvi


def get_scvi_metrics(adata, query_stu, bk, lk, dataset):
    ref = adata[~adata.obs[bk].isin(query_stu)].copy()
    
    # 1) Setup AnnData for scVI
    #    No need for common genes step since ref_adata comes from adata
    scvi.model.SCVI.setup_anndata(ref, batch_key="study")
    
    # 2) Train model on reference
    model = scvi.model.SCVI(ref, n_latent=8)
    model.train(max_epochs=100)
    
    # Adapt model to whole adata
    query_model = scvi.model.SCVI.load_query_data(adata, model)
    query_model.train(1)
    adata.obsm["X_scvi"]  = query_model.get_latent_representation(adata)

    return save_metrics(adata, ['X_scvi'], dataset, bk, lk, True)

# sample code

In [ ]:
X_scproto = tr.encode_adata(tr.dataset.adata, model)
X_scproto.shape

In [ ]:
adata = tr.dataset.adata
adata.obsm['X_scproto'] = X_scproto.detach().cpu().numpy()

In [ ]:
res = get_scib(adata, ['X_scproto', 'X_pca'], 'study', 'cell_type')
res

In [ ]:
# get scGraph metrics

In [ ]:
sys.path.append("/home/icb/fatemehs.hashemig/Islander/src")
from scGraph import *

In [ ]:
adata.write('immune-tmp.h5ad')

In [ ]:
scgraph = scGraph(
            adata_path='immune-tmp.h5ad',
            batch_key="study",
            label_key="cell_type",
        )

In [ ]:
scgr_res = scgraph.main(_obsm_list=['X_scproto', 'X_pca'])
scgr_res

In [ ]:
adata

# load t1, soft - hope for better scib

In [ ]:
params = {
    'cvae_loss_scaler': 0.01,
    'propagation_reg': 1,
    'experiment_name': 't1',
    'hard_clustering': 0
}

t1_soft = SwAV(**params)

In [ ]:
t1_soft.setup()


In [ ]:
model = t1_soft.get_model()

In [ ]:
path = '/home/icb/fatemehs.hashemig/models/pbmc-immune/t1_cvae_0.01_prop_1_hard_0/checkpoint.pth.tar'

checkpoint = torch.load(path)
model.load_state_dict(checkpoint["state_dict"])
model.to(self.device)

In [ ]:
adata.obsm['scProto_t1_soft'] = t1_soft.encode_adata(adata, model)

In [ ]:
adata.obsm['scProto_t1_soft'] = adata.obsm['scProto_t1_soft'].detach().cpu().numpy()

In [ ]:
res = get_scib(adata, ['X_scproto', 'X_pca', 'scProto_t1_soft'], 'study', 'cell_type')
res

In [ ]:
adata.write('immune-tmp.h5ad')
scgraph = scGraph(
            adata_path='immune-tmp.h5ad',
            batch_key="study",
            label_key="cell_type",
        )
scgr_res = scgraph.main(_obsm_list=['X_scproto', 'X_pca', 'scProto_t1_soft'])
scgr_res

In [ ]:
um = t1_soft.plot_umap(model, adata, None, False)

# baselines

In [ ]:
# train scvi, finetune only 1 epoch on query, get scib metrics

In [ ]:
path = '/home/icb/fatemehs.hashemig/data/scpoli/Immune_ALL_human_hvg.h5ad'
adata = sc.read_h5ad(path)

In [ ]:
import scvi
test_studies = ["Freytag", "Villani"]
test_ind = adata.obs.study.isin(test_studies)
ref = adata[~test_ind].copy()

# 1) Setup AnnData for scVI
#    No need for common genes step since ref_adata comes from adata
scvi.model.SCVI.setup_anndata(ref, batch_key="study")

# 2) Train model on reference
model = scvi.model.SCVI(ref, n_latent=8)
model.train(max_epochs=100)

In [ ]:
# Adapt model to whole adata
query_model = scvi.model.SCVI.load_query_data(adata, model)
query_model.train(1)
adata.obsm["X_scvi"]  = query_model.get_latent_representation(adata)

In [ ]:
adata.obsm.keys()

In [ ]:
def load_model(model):
    path = '/home/icb/fatemehs.hashemig/models/pbmc-immune/t1_cvae_0.01_prop_1_hard_0/checkpoint.pth.tar'

    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint["state_dict"])
    model.to(self.device)
    return model

In [ ]:
scproto_model = t1_soft.get_model()
scproto_model = load_model(scproto_model)

adata.obsm['scProto_t1_soft'] = t1_soft.encode_adata(adata, scproto_model).detach().cpu().numpy()

In [ ]:
adata.obsm.keys()

In [ ]:
sc.pp.pca(adata)

In [ ]:
emb_list = ['X_pca', 'scProto_t1_soft', 'X_scvi']

In [ ]:
res = get_scib(adata, ['X_pca', 'scProto_t1_soft', 'X_scvi'], 'study', 'final_annotation')
res

In [ ]:
adata.write('immune-tmp.h5ad')
scgraph = scGraph(
            adata_path='immune-tmp.h5ad',
            batch_key="study",
            label_key="final_annotation",
        )
scgr_res = scgraph.main(_obsm_list=emb_list)
scgr_res

In [ ]:
# debug pancreas, init swav with pancreas, check train and test ds size and their loaders
# check train and test epochs

In [ ]:
pt = SwAV(dataset_id = 'pancreas')

In [ ]:
pt.setup()

In [ ]:
pt.batch_size = 128
pt.setup()

In [ ]:
len(pt.train_loader), len(pt.test_loader)

In [ ]:
pt.train(3)

# why even in 1 ds setting, it cannnot achive high matching pairs?

In [ ]:
t = SwAV(debug=1, study_id = '10X', num_prototypes=150)

In [ ]:
t.setup()

In [ ]:
self = t.train_ds
self.save_path

In [ ]:
!rm ./graphs/graph_pbmc-immune9654_pca50_knn50.pkl

In [ ]:
os.path.exists(self.save_path)

In [ ]:
adata = t.dataset.adata

In [ ]:
from interpretable_ssl.augmenters.graph_utils import *

In [ ]:
adata.obs.keys()

In [ ]:
ind, dist = faiss_knn_within_batches(adata, 'study', 50, 50)

In [ ]:
adata.obsm

In [ ]:
# goal: calc matching pairs ratio in center of pca kmeans scenario

# calc pca in the 1 ds
# calc kmeans and get centers
# for each sample and all of the neighbors, calc if they map in 1 center or not

In [ ]:
adata = t.ref.adata

In [ ]:
sc.tl.pca(adata, n_comps = 50)

In [ ]:
adata.obsm.keys()

In [ ]:
from sklearn.cluster import KMeans

# Assuming adata already has PCA stored
X = adata.obsm['X_pca']

# Choose number of clusters (prototypes)
n_clusters = 50  # adjust as needed

kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init="auto").fit(X)
# Prototypes (kmeans centers)
prototypes = kmeans.cluster_centers_

In [ ]:
# Compute dot products: (n_cells, n_clusters)
dot_products = X @ prototypes.T

# Assign each cell to the prototype with max dot product
assignments = np.argmax(dot_products, axis=1)

# Store in adata.obs
adata.obs['dot_proto'] = assignments.astype(str)


In [ ]:
k = 50
batch_pca = adata.obsm['X_pca']

index = faiss.IndexFlatL2(batch_pca.shape[1])
index.add(batch_pca)
D, I = index.search(batch_pca, k + 1)

for each sample
check how many of the neighbors are in same prototype
report avg matching pairs

In [ ]:
neighbors = I[:, 1:]  # shape (n_cells, k)

# Make sure labels are ints
labels = adata.obs['dot_proto'].astype(int).to_numpy()  # shape (n_cells,)

# Labels of each cell's neighbors
neighbor_labels = labels[neighbors]  # shape (n_cells, k)

# Compare neighbor labels to the cell's own label
same_mask = (neighbor_labels == labels[:, None])  # bool matrix

# Count matches per cell and average
matches_per_cell = same_mask.sum(axis=1)          # shape (n_cells,)
avg_matching_count = matches_per_cell.mean()      # average count out of k
avg_matching_fraction = (matches_per_cell / k).mean()  # average fraction

print(f"Avg matching neighbors per cell: {avg_matching_count:.3f} / {k}")
print(f"Avg matching fraction: {avg_matching_fraction:.4f}")

In [ ]:
len(adata)

In [ ]:
pwd

In [ ]:
path = '/ictstr01/home/icb/fatemehs.hashemig/data/seacell/cd34_multiome_rna_preprocessed.h5ad'

In [ ]:
adata = sc.read_h5ad(path)
adata.obs.head()

In [ ]:
from interpretable_ssl.trainers.scpoli_original import *

In [ ]:
t = OriginalTrainer(dataset_id = 'pancreas')

In [ ]:
t.get_model_path()

# scProto: scIB, scGraph metrics 

In [ ]:
params = {
    'cvae_loss_scaler': 0.01,
    'propagation_reg': 1,
    'experiment_name': 't2',
}

## pancreas

In [ ]:
pancreas = {"dataset_id": "pancreas", "num_prototypes": 220, "batch_size": 128}
t = SwAV(debug = 1, **(params|pancreas))

In [ ]:
t.setup()

In [ ]:
metrics, scgraph = save_metrics(t, 'pancreas', False)

### calc and save scGraph

In [ ]:
get_scgraph(adata, [t.get_model_name(), 'X_pca'], 'pancreas')

## immune

In [ ]:
params['experiment_name'] = 't1'
t = SwAV(debug = 1, **params)

In [ ]:
t.setup()

In [ ]:
save_metrics(t, 'pbmc-immune')

# scPoli

In [ ]:
from interpretable_ssl.trainers.scpoli_original import *

## immune

In [ ]:
t = OriginalTrainer(debug=1)

In [ ]:
t.dataset.adata

In [ ]:
save_metrics(t, 'pbmc-immune', True)

## pancreas

In [ ]:
t = OriginalTrainer(debug=1, dataset_id = 'pancreas')

In [ ]:
t.setup()

In [ ]:
save_metrics(t, 'pancreas', True)

# harmoney and seacell

In [ ]:
pwd

In [ ]:
adata = sc.read_h5ad('/ictstr01/home/icb/fatemehs.hashemig//data/scpoli/Immune_ALL_human_hvg.h5ad')

adata

In [ ]:
sc.tl.pca(adata)

In [ ]:
import scanpy as sc
import scanpy.external as sce

# 2) Run Harmony batch correction
sce.pp.harmony_integrate(
    adata,
    key="study",                     # your batch column
    basis="X_pca",                   # which embedding to correct
    adjusted_basis="X_pca_harmoney"   # where to store corrected PCs
)


In [ ]:
adata

In [ ]:
save_metrics(adata, ['X_pca', 'X_pca_harmoney'], 'pbmc-immune', 'study', 'final_annotation', True)

In [ ]:
import scanpy as sc
import scanpy.external as sce

def save_pca_harmoney_metrics(adata, bk, lk, dataset):
    sc.tl.pca(adata)
    # 2) Run Harmony batch correction
    sce.pp.harmony_integrate(
        adata,
        key="study",                     # your batch column
        basis="X_pca",                   # which embedding to correct
        adjusted_basis="X_pca_harmoney"   # where to store corrected PCs
    )
    return save_metrics(adata, ['X_pca', 'X_pca_harmoney'], dataset, bk, lk, True)

## pancreas

In [ ]:
home = '/ictstr01/home/icb/fatemehs.hashemig/'
adata = sc.read_h5ad(f'{home}/data/scpoli/pancreas_sparse-pca.h5ad')
adata

In [ ]:
scib_m, scgraph_m = save_pca_harmoney_metrics(adata, 'study', 'cell_type', 'pancreas')

In [ ]:
scib_m

In [ ]:
scgraph_m

# scvi

In [ ]:
get_scvi_metrics(adata, ["celseq", "celseq2"], 'study', 'cell_type', 'pancreas')

## immune

In [ ]:
adata = sc.read_h5ad('/ictstr01/home/icb/fatemehs.hashemig//data/scpoli/Immune_ALL_human_hvg.h5ad')
scib_m, scgraph_m = get_scvi_metrics(adata, ["Freytag", "Villani"], 'study', 'final_annotation', 'pbmc-immune')

In [ ]:
scib_m

In [ ]:
scib_m = scib_m.reset_index().drop_duplicates().set_index("Embedding")


In [ ]:
scgraph_m

# seacell

In [ ]:
import SEACells

In [ ]:
def compute_seacells(ad, n_SEACells, build_kernel_on = 'X_pca'):

  ## Additional parameters
  n_waypoint_eigs = 10 # Number of eigenvalues to consider when initializing metacells

  model = SEACells.core.SEACells(ad,
                  build_kernel_on=build_kernel_on,
                  n_SEACells=n_SEACells,
                  n_waypoint_eigs=n_waypoint_eigs,
                  convergence_epsilon = 1e-5)

  model.construct_kernel_matrix()
  M = model.kernel_matrix
  # Initialize archetypes
  model.initialize_archetypes()
  model.fit(min_iter=10, max_iter=50)

  SEACell_ad = SEACells.core.summarize_by_SEACell(ad, SEACells_label='SEACell', summarize_layer='raw')
  # SEACell_soft_ad = SEACells.core.summarize_by_soft_SEACell(ad, model.A_, celltype_label='celltype',summarize_layer='raw', minimum_weight=0.05)
  return ad, SEACell_ad, model

In [ ]:
def agg_obs(SEACell_ad, adata, obs_key):
    SEACell_ad.obs[obs_key] = (
    adata.obs
    .groupby('SEACell')[obs_key]
    .agg(lambda x: x.mode()[0])
    .reindex(SEACell_ad.obs_names)
    )
    return SEACell_ad

def get_seacell_metrics(SEACell_ad, adata, bk, lk, ds):
  # Normalize cells, log transform and compute highly variable genes
  sc.pp.normalize_per_cell(SEACell_ad)
  sc.pp.log1p(SEACell_ad)
  # sc.pp.highly_variable_genes(ad, n_top_genes=1500)
  sc.tl.pca(SEACell_ad, n_comps=50, obsm='seacell_pca')

  sce.pp.harmony_integrate(
    SEACell_ad,
    key=bk,                     # your batch column
    basis="seacell_pca",                   # which embedding to correct
    adjusted_basis="seacell_pca_harmoney"   # where to store corrected PCs
    )
  SEACell_ad = agg_obs(SEACell_ad, adata, bk)
  SEACell_ad = agg_obs(SEACell_ad, adata, lk)
    
  return save_metrics(SEACell_ad, ['seacell_pca', 'seacell_pca_harmoney'], ds, bk, lk, True)

## immune

In [ ]:
sc.tl.pca(adata)

In [ ]:
ad, SEACell_ad, model = compute_seacells(adata, 300)

In [ ]:
get_seacell_metrics(SEACell_ad, ad, 'study', 'final_annotation', 'pbmc-immune')

## pancreas

In [ ]:
def load_pancreas():
    home = '/ictstr01/home/icb/fatemehs.hashemig/'
    adata = sc.read_h5ad(f'{home}/data/scpoli/pancreas_sparse-pca.h5ad')
    sc.tl.pca(adata)
    return adata

adata = load_pancreas()

In [ ]:
ad, SEACell_ad, model = compute_seacells(adata, 220)

In [ ]:
get_seacell_metrics(SEACell_ad, ad, 'study', 'cell_type', 'pancreas')